# Range Query Data Structures

Data structures optimized for range queries and point updates on arrays.

## Topics Covered
- Prefix Sum (Range Sum Queries - Static)
- Difference Array (Range Updates - Static)
- Segment Tree (Build, Query, Update)
- Fenwick Tree / Binary Indexed Tree (Prefix Sum, Point Update, Range Query)
- Sparse Table (Range Minimum Query)

---
## 1. Prefix Sum

For static arrays, prefix sum enables O(1) range sum queries after O(n) preprocessing.

In [ ]:
class PrefixSum:
    """Prefix sum for O(1) range sum queries on static array."""
    def __init__(self, arr):
        n = len(arr)
        self.prefix = [0] * (n + 1)
        for i in range(n):
            self.prefix[i + 1] = self.prefix[i] + arr[i]

    def query(self, l, r):
        """Sum of arr[l..r] (0-indexed)."""
        return self.prefix[r + 1] - self.prefix[l]

ps = PrefixSum([3, 1, 4, 1, 5, 9, 2, 6])
print(f"Sum [0..3]: {ps.query(0, 3)}")  # 9
print(f"Sum [2..5]: {ps.query(2, 5)}")  # 19
print(f"Sum [4..7]: {ps.query(4, 7)}")  # 22
# Build: O(n), Query: O(1), Space: O(n)

---
## 2. Difference Array

For range add updates on static arrays. O(1) per update, O(n) to reconstruct final array.

In [ ]:
class DifferenceArray:
    """Efficient range add updates."""
    def __init__(self, arr):
        self.n = len(arr)
        self.diff = [0] * (self.n + 1)
        # Build diff from initial array
        for i, val in enumerate(arr):
            self.add(i, i, val)

    def add(self, l, r, val):
        """Add val to all elements in arr[l..r]."""
        self.diff[l] += val
        self.diff[r + 1] -= val

    def apply(self):
        """Apply all updates and return the final array."""
        result = [0] * self.n
        curr = 0
        for i in range(self.n):
            curr += self.diff[i]
            result[i] = curr
        return result

da = DifferenceArray([0, 0, 0, 0, 0])
da.add(0, 2, 5)
da.add(1, 3, 3)
da.add(2, 4, -2)
print(f"After updates: {da.apply()}")  # [5, 8, 6, 1, -2]
# Update: O(1), Apply: O(n)

---
## 3. Segment Tree

A tree data structure for range queries and point updates in O(log n). Each node represents a segment of the array.

In [ ]:
class SegmentTree:
    """Segment tree for range sum queries with point updates."""
    def __init__(self, arr):
        self.n = len(arr)
        self.tree = [0] * (4 * self.n)
        self._build(arr, 0, 0, self.n - 1)

    def _build(self, arr, node, l, r):
        if l == r:
            self.tree[node] = arr[l]
            return
        mid = (l + r) // 2
        self._build(arr, 2 * node + 1, l, mid)
        self._build(arr, 2 * node + 2, mid + 1, r)
        self.tree[node] = self.tree[2 * node + 1] + self.tree[2 * node + 2]

    def update(self, idx, val):
        """Set arr[idx] = val."""
        self._update(0, 0, self.n - 1, idx, val)

    def _update(self, node, l, r, idx, val):
        if l == r:
            self.tree[node] = val
            return
        mid = (l + r) // 2
        if idx <= mid:
            self._update(2 * node + 1, l, mid, idx, val)
        else:
            self._update(2 * node + 2, mid + 1, r, idx, val)
        self.tree[node] = self.tree[2 * node + 1] + self.tree[2 * node + 2]

    def query(self, ql, qr):
        """Return sum of arr[ql..qr]."""
        return self._query(0, 0, self.n - 1, ql, qr)

    def _query(self, node, l, r, ql, qr):
        if ql <= l and r <= qr:
            return self.tree[node]
        if r < ql or l > qr:
            return 0
        mid = (l + r) // 2
        left = self._query(2 * node + 1, l, mid, ql, qr)
        right = self._query(2 * node + 2, mid + 1, r, ql, qr)
        return left + right

st = SegmentTree([1, 3, 5, 7, 9, 11])
print(f"Sum [1..3]: {st.query(1, 3)}")  # 15
st.update(2, 10)
print(f"After update, sum [1..3]: {st.query(1, 3)}")  # 20
# Time: O(log n) per operation, Space: O(n)

---
## 4. Fenwick Tree (Binary Indexed Tree)

BIT supports prefix sum queries and point updates in O(log n). Simpler and more space-efficient than segment tree.

In [ ]:
class FenwickTree:
    """Binary Indexed Tree for prefix sums and point updates."""
    def __init__(self, n):
        self.n = n
        self.bit = [0] * (n + 1)

    def build(self, arr):
        """Build BIT from array."""
        for i, val in enumerate(arr):
            self.add(i, val)

    def add(self, idx, delta):
        """Add delta to arr[idx]."""
        i = idx + 1  # BIT is 1-indexed
        while i <= self.n:
            self.bit[i] += delta
            i += i & -i  # Move to parent

    def prefix_sum(self, idx):
        """Sum of arr[0..idx]."""
        i = idx + 1
        result = 0
        while i > 0:
            result += self.bit[i]
            i -= i & -i  # Move to previous
        return result

    def range_sum(self, l, r):
        """Sum of arr[l..r]."""
        return self.prefix_sum(r) - self.prefix_sum(l - 1)

    def update(self, idx, new_val, arr):
        """Set arr[idx] = new_val."""
        delta = new_val - arr[idx]
        arr[idx] = new_val
        self.add(idx, delta)

arr = [1, 3, 5, 7, 9, 11]
bit = FenwickTree(len(arr))
bit.build(arr)
print(f"Prefix sum [0..3]: {bit.prefix_sum(3)}")  # 16
print(f"Range sum [1..3]: {bit.range_sum(1, 3)}")  # 15
bit.add(2, 5)  # Add 5 to index 2
print(f"After add, range sum [1..3]: {bit.range_sum(1, 3)}")  # 20
# Time: O(log n) per operation, Space: O(n)

---
## 5. Sparse Table

For idempotent range queries (min, max, gcd) on static arrays. O(1) query after O(n log n) preprocessing.

In [ ]:
import math

class SparseTable:
    """Sparse table for range minimum queries (RMQ)."""
    def __init__(self, arr):
        self.n = len(arr)
        self.log = [0] * (self.n + 1)
        for i in range(2, self.n + 1):
            self.log[i] = self.log[i // 2] + 1
        k = self.log[self.n] + 1
        self.st = [[0] * k for _ in range(self.n)]
        for i in range(self.n):
            self.st[i][0] = arr[i]
        j = 1
        while (1 << j) <= self.n:
            i = 0
            while i + (1 << j) <= self.n:
                self.st[i][j] = min(self.st[i][j-1],
                                    self.st[i + (1 << (j-1))][j-1])
                i += 1
            j += 1

    def query(self, l, r):
        """Return minimum in arr[l..r] in O(1)."""
        j = self.log[r - l + 1]
        return min(self.st[l][j], self.st[r - (1 << j) + 1][j])

arr = [4, 2, 3, 1, 7, 5, 8, 6]
st = SparseTable(arr)
print(f"Min [1..4]: {st.query(1, 4)}")  # 1
print(f"Min [2..5]: {st.query(2, 5)}")  # 1
print(f"Min [0..2]: {st.query(0, 2)}")  # 2
# Build: O(n log n), Query: O(1), Space: O(n log n)

---
## 6. Performance Comparison

| Structure | Build | Query | Update | Space | Best For |
|---|---|---|---|---|---|
| Prefix Sum | O(n) | O(1) | N/A (static) | O(n) | Static range sum |
| Difference Array | O(n) | O(n) | O(1) range | O(n) | Range add, single read |
| Segment Tree | O(n) | O(log n) | O(log n) | O(n) | Dynamic range sum |
| Fenwick Tree | O(n) | O(log n) | O(log n) | O(n) | Prefix sums, point updates |
| Sparse Table | O(n log n) | O(1) | N/A (static) | O(n log n) | Static RMQ |

---
## LeetCode Problems

### LeetCode 303 - Range Sum Query - Immutable (Prefix Sum)

**Problem:** Implement NumArray that supports range sum queries on a static array.

In [ ]:
class NumArray:
    def __init__(self, nums):
        self.prefix = [0]
        for num in nums:
            self.prefix.append(self.prefix[-1] + num)

    def sumRange(self, left, right):
        return self.prefix[right + 1] - self.prefix[left]

na = NumArray([-2, 0, 3, -5, 2, -1])
print(na.sumRange(0, 2))  # 1
print(na.sumRange(2, 5))  # -1
print(na.sumRange(0, 5))  # -3
# Time: O(1) per query, O(n) build

### LeetCode 307 - Range Sum Query - Mutable (Segment Tree / BIT)

**Problem:** NumArray with point updates and range sum queries.

In [ ]:
class NumMutableArray:
    """Using Fenwick Tree."""
    def __init__(self, nums):
        self.nums = nums
        self.n = len(nums)
        self.bit = [0] * (self.n + 1)
        for i, val in enumerate(nums):
            self._add(i, val)

    def _add(self, idx, delta):
        i = idx + 1
        while i <= self.n:
            self.bit[i] += delta
            i += i & -i

    def _prefix(self, idx):
        i = idx + 1
        result = 0
        while i > 0:
            result += self.bit[i]
            i -= i & -i
        return result

    def update(self, index, val):
        delta = val - self.nums[index]
        self.nums[index] = val
        self._add(index, delta)

    def sumRange(self, left, right):
        return self._prefix(right) - self._prefix(left - 1)

nma = NumMutableArray([1, 3, 5])
print(nma.sumRange(0, 2))  # 9
nma.update(1, 2)
print(nma.sumRange(0, 2))  # 8
# Time: O(log n) per operation

### LeetCode 218 - The Skyline Problem (Sweep Line with Segment Tree concepts)

**Problem:** Return skyline formed by buildings using sweep line.

In [ ]:
import heapq

def getSkyline(buildings):
    """Return skyline key points."""
    events = []
    for l, r, h in buildings:
        events.append((l, -h, r))  # Start: negative height for sorting
        events.append((r, 0, 0))   # End: height 0
    events.sort()

    result = []
    heap = [(0, float('inf'))]  # (height, end position)
    prev_height = 0
    for x, neg_h, r in events:
        if neg_h != 0:  # Building starts
            heapq.heappush(heap, (neg_h, r))
        while heap and heap[0][1] <= x:
            heapq.heappop(heap)
        curr_height = -heap[0][0]
        if curr_height != prev_height:
            result.append([x, curr_height])
            prev_height = curr_height
    return result

buildings = [[2,9,10],[3,7,15],[5,12,12],[15,20,10],[19,24,8]]
print(f"Skyline: {getSkyline(buildings)}")
# Time: O(n log n), Space: O(n)

---
## Summary

When to use each structure:

| Scenario | Best Structure |
|---|---|
| Static array, many range sum queries | Prefix Sum |
| Static array, range minimum queries | Sparse Table |
| Dynamic array, point updates + range sum | Fenwick Tree (if prefix sum) or Segment Tree |
| Range add updates, single final read | Difference Array |
| Complex range queries (sum, min, max) | Segment Tree |
| 2D queries | 2D Fenwick / Segment Tree |